# **IMBALANCE ANALYSIS AND MITIGATION**

## **IMPORTS**

In [33]:
# General imports
import pandas as pd
import numpy as np
from imblearn.pipeline import make_pipeline as imb_make_pipeline
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, cohen_kappa_score

# Preprocessor imports
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from preprocessing.preprocessor import preprocessorLOEEALL

# Oversampling imports
from imblearn.over_sampling import SMOTE, RandomOverSampler, ADASYN, BorderlineSMOTE

# Undersampling imports
from imblearn.under_sampling import RandomUnderSampler, EditedNearestNeighbours, NearMiss, TomekLinks

# Combination imports
from imblearn.combine import SMOTEENN, SMOTETomek

# Model imports
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, BaggingClassifier
import xgboost as xgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

## **DATASET LOADING**

In [34]:
# We read the partitions
X_train = pd.read_parquet("../data/cleaned/X_train.parquet")
X_test = pd.read_parquet("../data/cleaned/X_test.parquet")
y_train = pd.read_parquet("../data/cleaned/y_train.parquet").squeeze()
y_test = pd.read_parquet("../data/cleaned/y_test.parquet").squeeze()

# Check that all has been loaded correctly
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

categorical_cols = ['Type', 'Gender', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'RescuerID', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength','Breed1', 'Breed2', 'State']

(10045, 32)
(10045,)
(4948, 32)
(4948,)


## **MODELS & TECHNIQUES**

### Models

In [35]:
# MODELS: default hyperparameters, random_state=42
MODELS = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Bagging': BaggingClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0, allow_writing_files=False)
}

### Balanced models

In [36]:
# MODELS_BALANCED: same models, class_weight='balanced' (where possible)
MODELS_BALANCED = {
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'AdaBoost': AdaBoostClassifier(random_state=42),  # doesn't have class_weight
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),  # doesn't have class_weight 
    'Bagging': BaggingClassifier(random_state=42),  # doesn't have class_weight 
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),  # doesn't have class_weight
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced'),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0, auto_class_weights='Balanced', allow_writing_files=False),
    'CatBoost_native_categorical': CatBoostClassifier(random_state=42, verbose=0, allow_writing_files=False, cat_features=categorical_cols),
    'CatBoost_native_categorical_balanced': CatBoostClassifier(random_state=42, verbose=0, allow_writing_files=False, auto_class_weights='Balanced', cat_features=categorical_cols)
}

### Balancing techniques

In [37]:
# BALANCING_TECHNIQUES: techniques to mitigate the class imbalance
BALANCING_TECHNIQUES = {
    'None': 'passthrough',  # to compare without resampling
    'SMOTE (oversampling)': SMOTE(random_state=42),
    'RandomOverSampler (oversampling)': RandomOverSampler(random_state=42),
    'ADASYN (oversampling)': ADASYN(random_state=42, sampling_strategy='minority'),
    'BorderlineSMOTE (oversampling)': BorderlineSMOTE(random_state=42),
    'RandomUnderSampler (undersampling)': RandomUnderSampler(random_state=42),
    'EditedNearestNeighbours (undersampling)': EditedNearestNeighbours(),
    'NearMiss (undersampling)': NearMiss(),
    'TomekLinks (undersampling)': TomekLinks(),
    'SMOTEENN (combination)': SMOTEENN(random_state=42),
    'SMOTETomek (combination)': SMOTETomek(random_state=42)
}

### Initialization

In [38]:
# Initializing the folds that we will be using
skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

# Initializng the second evaluation technique
qwk = make_scorer(cohen_kappa_score, weights="quadratic")
scoring = {
    "f1_macro": "f1_macro",
    "balanced_acc": "balanced_accuracy",
    "mcc": "matthews_corrcoef",
    "QWK": qwk
}

# Initializing the results
results = []

## **IMBALANCE ANALYSIS**

In [39]:
target_distribution_train = y_train.value_counts()
target_distribution_test = y_test.value_counts()


for key, amount_train, amount_test in zip(target_distribution_train.index, target_distribution_train.values, target_distribution_test):
    print(f"""Class {key}: 
        Amount train: {amount_train} --> {round((amount_train/y_train.shape[0])*100, 2)}%
        Amount test: {amount_test} --> {round((amount_test/y_test.shape[0])*100, 2)}%""")


Class 4: 
        Amount train: 2812 --> 27.99%
        Amount test: 1385 --> 27.99%
Class 2: 
        Amount train: 2705 --> 26.93%
        Amount test: 1332 --> 26.92%
Class 3: 
        Amount train: 2183 --> 21.73%
        Amount test: 1076 --> 21.75%
Class 1: 
        Amount train: 2070 --> 20.61%
        Amount test: 1020 --> 20.61%
Class 0: 
        Amount train: 275 --> 2.74%
        Amount test: 135 --> 2.73%


## **IMBALANCE MITIGATION**

### Base models

In [ ]:
# First we execute the base models

for model_name, model in MODELS_BALANCED.items():

    if model_name != "CatBoost_native_categorical" and model_name != "CatBoost_native_categorical_balanced":
        pipe = imb_make_pipeline(preprocessorLOEEALL, model)
    else:
        pipe = imb_make_pipeline(model)

    results_cv = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1, return_train_score=True)

    result = {
        "Model": f"{model_name} balanced",
        "balancing_technique": "no",

        # F1 Macro
        "f1_mean_test": np.mean(results_cv["test_f1_macro"]),
        "f1_mean_train": np.mean(results_cv["train_f1_macro"]),
        "f1_sd_test": np.std(results_cv["test_f1_macro"]),
        "f1_sd_train": np.std(results_cv["train_f1_macro"]),

        # QWK (Quadratic Weighted Kappa)
        "qwk_mean_test": np.mean(results_cv["test_QWK"]),
        "qwk_mean_train": np.mean(results_cv["train_QWK"]),
        "qwk_sd_test": np.std(results_cv["test_QWK"]),
        "qwk_sd_train": np.std(results_cv["train_QWK"]),

        # Balanced Accuracy
        "balanced_acc_mean_test": np.mean(results_cv["test_balanced_acc"]),
        "balanced_acc_mean_train": np.mean(results_cv["train_balanced_acc"]),
        "balanced_acc_sd_test": np.std(results_cv["test_balanced_acc"]),
        "balanced_acc_sd_train": np.std(results_cv["train_balanced_acc"]),

        # MCC (Matthews Correlation Coefficient)
        "mcc_mean_test": np.mean(results_cv["test_mcc"]),
        "mcc_mean_train": np.mean(results_cv["train_mcc"]),
        "mcc_sd_test": np.std(results_cv["test_mcc"]),
        "mcc_sd_train": np.std(results_cv["train_mcc"])
    }
    results.append(result)

    print(f"{model_name} -> F1-macro: {round(result['f1_mean'],4)}, "
        f"QWK: {round(result['qwk_mean'],4)}, "
        f"Balanced Acc: {round(result['balanced_acc_mean'],4)}, "
        f"MCC: {round(result['mcc_mean'],4)}")


Random Forest -> F1-macro: 0.3123, QWK: 0.3954, Balanced Acc: 0.3281, MCC: 0.226
AdaBoost -> F1-macro: 0.299, QWK: 0.3574, Balanced Acc: 0.3153, MCC: 0.2021
Gradient Boosting -> F1-macro: 0.3119, QWK: 0.4004, Balanced Acc: 0.329, MCC: 0.2285
Bagging -> F1-macro: 0.2949, QWK: 0.3479, Balanced Acc: 0.3042, MCC: 0.1802
XGBoost -> F1-macro: 0.3089, QWK: 0.3756, Balanced Acc: 0.3209, MCC: 0.208
LightGBM -> F1-macro: 0.3293, QWK: 0.395, Balanced Acc: 0.3385, MCC: 0.2177
CatBoost -> F1-macro: 0.3289, QWK: 0.393, Balanced Acc: 0.338, MCC: 0.2234
CatBoost_native_categorical -> F1-macro: 0.3543, QWK: 0.4396, Balanced Acc: 0.3622, MCC: 0.2704
CatBoost_native_categorical_balanced -> F1-macro: 0.3614, QWK: 0.4329, Balanced Acc: 0.3701, MCC: 0.2604


In [41]:
# Save partial results to a csv
partial_results = pd.DataFrame(results)

partial_results.to_csv("technique_results/partial_results.csv")



### Models & balancing techniques

In [ ]:
# We check all the combinations of balancing techniques and models
#! DON'T EXECUTE AGAIN (+1 hour) --> with n_jobs=-1 15 minutes

for technique_name, technique in BALANCING_TECHNIQUES.items():
    for model_name, model in MODELS.items():
        pipe = imb_make_pipeline(preprocessorLOEEALL, technique, model)

        results_cv = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1, return_train_score=True)

        result = {
            "Model": f"{model_name}",
            "balancing_technique": f"{technique_name}",

            # F1 Macro
            "f1_mean_test": np.mean(results_cv["test_f1_macro"]),
            "f1_mean_train": np.mean(results_cv["train_f1_macro"]),
            "f1_sd_test": np.std(results_cv["test_f1_macro"]),
            "f1_sd_train": np.std(results_cv["train_f1_macro"]),

            # QWK (Quadratic Weighted Kappa)
            "qwk_mean_test": np.mean(results_cv["test_QWK"]),
            "qwk_mean_train": np.mean(results_cv["train_QWK"]),
            "qwk_sd_test": np.std(results_cv["test_QWK"]),
            "qwk_sd_train": np.std(results_cv["train_QWK"]),

            # Balanced Accuracy
            "balanced_acc_mean_test": np.mean(results_cv["test_balanced_acc"]),
            "balanced_acc_mean_train": np.mean(results_cv["train_balanced_acc"]),
            "balanced_acc_sd_test": np.std(results_cv["test_balanced_acc"]),
            "balanced_acc_sd_train": np.std(results_cv["train_balanced_acc"]),

            # MCC (Matthews Correlation Coefficient)
            "mcc_mean_test": np.mean(results_cv["test_mcc"]),
            "mcc_mean_train": np.mean(results_cv["train_mcc"]),
            "mcc_sd_test": np.std(results_cv["test_mcc"]),
            "mcc_sd_train": np.std(results_cv["train_mcc"])
        }
        results.append(result)

        print(f"{model_name} using {technique_name} -> F1-macro: {round(result['f1_mean'],4)}, "
            f"QWK: {round(result['qwk_mean'],4)}, "
            f"Balanced Acc: {round(result['balanced_acc_mean'],4)}, "
            f"MCC: {round(result['mcc_mean'],4)}")




Random Forest using None -> F1-macro: 0.3114, QWK: 0.3883, Balanced Acc: 0.3296, MCC: 0.231
AdaBoost using None -> F1-macro: 0.299, QWK: 0.3574, Balanced Acc: 0.3153, MCC: 0.2021
Gradient Boosting using None -> F1-macro: 0.3119, QWK: 0.4004, Balanced Acc: 0.329, MCC: 0.2285
Bagging using None -> F1-macro: 0.2949, QWK: 0.3479, Balanced Acc: 0.3042, MCC: 0.1802
XGBoost using None -> F1-macro: 0.3089, QWK: 0.3756, Balanced Acc: 0.3209, MCC: 0.208
LightGBM using None -> F1-macro: 0.3096, QWK: 0.3805, Balanced Acc: 0.3234, MCC: 0.2136
CatBoost using None -> F1-macro: 0.3115, QWK: 0.3913, Balanced Acc: 0.3274, MCC: 0.2231
Random Forest using SMOTE (oversampling) -> F1-macro: 0.3393, QWK: 0.3748, Balanced Acc: 0.3526, MCC: 0.2195
AdaBoost using SMOTE (oversampling) -> F1-macro: 0.2991, QWK: 0.3214, Balanced Acc: 0.3252, MCC: 0.1707
Gradient Boosting using SMOTE (oversampling) -> F1-macro: 0.3215, QWK: 0.3726, Balanced Acc: 0.3457, MCC: 0.2128
Bagging using SMOTE (oversampling) -> F1-macro: 0.

In [43]:
# Save results to a csv
final_results = pd.DataFrame(results)
final_results.to_csv("technique_results/final_results.csv")


### Extra for catboost native categorical

In [ ]:
from imblearn.over_sampling import SMOTENC, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Asumiendo que 'cat_features_indices' es una lista con los índices de tus columnas categóricas
BALANCING_TECHNIQUES_CATBOOST = {    
    # El único SMOTE que respeta categorías:
    'SMOTENC (oversampling)': SMOTENC(categorical_features=categorical_cols, random_state=42),
    # Seguro para categorías:
    'RandomOverSampler (oversampling)': RandomOverSampler(random_state=42),
    'RandomUnderSampler (undersampling)': RandomUnderSampler(random_state=42),
}

model_name = 'CatBoost_native_categorical'
model = CatBoostClassifier(random_state=42, verbose=0, allow_writing_files=False, cat_features=categorical_cols)

for technique_name, technique in BALANCING_TECHNIQUES_CATBOOST.items():
    pipe = imb_make_pipeline(preprocessorLOEEALL, technique, model)
    results_cv = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1, return_train_score=True)
    result = {
    "Model": f"{model_name}",
    "balancing_technique": f"{technique_name}",

    # F1 Macro
    "f1_mean_test": np.mean(results_cv["test_f1_macro"]),
    "f1_mean_train": np.mean(results_cv["train_f1_macro"]),
    "f1_sd_test": np.std(results_cv["test_f1_macro"]),
    "f1_sd_train": np.std(results_cv["train_f1_macro"]),

    # QWK (Quadratic Weighted Kappa)
    "qwk_mean_test": np.mean(results_cv["test_QWK"]),
    "qwk_mean_train": np.mean(results_cv["train_QWK"]),
    "qwk_sd_test": np.std(results_cv["test_QWK"]),
    "qwk_sd_train": np.std(results_cv["train_QWK"]),

    # Balanced Accuracy
    "balanced_acc_mean_test": np.mean(results_cv["test_balanced_acc"]),
    "balanced_acc_mean_train": np.mean(results_cv["train_balanced_acc"]),
    "balanced_acc_sd_test": np.std(results_cv["test_balanced_acc"]),
    "balanced_acc_sd_train": np.std(results_cv["train_balanced_acc"]),

    # MCC (Matthews Correlation Coefficient)
    "mcc_mean_test": np.mean(results_cv["test_mcc"]),
    "mcc_mean_train": np.mean(results_cv["train_mcc"]),
    "mcc_sd_test": np.std(results_cv["test_mcc"]),
    "mcc_sd_train": np.std(results_cv["train_mcc"])
}
results.append(result)

print(f"{model_name} using {technique_name} -> F1-macro: {round(result['f1_mean'],4)}, "
    f"QWK: {round(result['qwk_mean'],4)}, "
    f"Balanced Acc: {round(result['balanced_acc_mean'],4)}, "
    f"MCC: {round(result['mcc_mean'],4)}")

In [ ]:
# Save results to a csv
extra_results = pd.DataFrame(results)
extra_results.to_csv("technique_results/extra_results.csv")